# 3. Оценка и исследовательский отчёт

Оценка моделей выполняется на GPU-сервере. `RUN_NAME` и `CHECKPOINT_KIND` задают опыт и тип checkpoint. Каждый опыт оценивается после перезапуска kernel по одному протоколу.

Метрики WikiWhy ROUGE-L F1 и MMLU-Pro/history accuracy рассчитываются для base, Full FT, LoRA baseline и каждого варианта LoRA. Исторический экзамен оценивает перенос на тесты, ручная проверка — смысл причинных объяснений.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import random
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from data import make_prompt, make_target
from training import MODEL_NAME, MODEL_REVISION, load_base_model
from evaluation import make_history_messages, generate_answers, rouge_l, parse_choice, history_accuracy

ROOT = Path.cwd()
assert (ROOT / 'evaluation.py').is_file(), 'Рабочая папка notebook должна быть hw4_lora'
assert torch.cuda.is_available(), 'Для оценки моделей требуется CUDA'
DEVICE = 'cuda'
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
wikiwhy_test = [json.loads(line) for line in (ROOT / 'data/wikiwhy/test.jsonl').read_text().splitlines()]
history_test = [json.loads(line) for line in (ROOT / 'data/mmlu_history/test.jsonl').read_text().splitlines()]
print('WikiWhy:', len(wikiwhy_test), 'MMLU-Pro/history:', len(history_test))

## Один протокол генерации

Оценка выполняется на полном test. Отладочная генерация проводится на validation. Исторический test не участвует в выборе настроек.

Эталон WikiWhy и `cot_content` MMLU-Pro не входят в messages. Для всех моделей используется один режим генерации с отключённым sampling.

In [ ]:
GENERATION_CONFIG = {'max_new_tokens': 384, 'do_sample': False}
BATCH_SIZE = 1
wikiwhy_prompts = [make_prompt(row) for row in wikiwhy_test]
history_prompts = [make_history_messages(row) for row in history_test]
wikiwhy_references = [make_target(row) for row in wikiwhy_test]
print('WikiWhy prompt:', wikiwhy_prompts[0])
print('History prompt:', history_prompts[0])

## Загрузка выбранной модели

`RUN_NAME` задаёт имя опыта, `CHECKPOINT_KIND` — тип: `base`, `full_ft` или `lora`. Каталог обученной модели `runs/<RUN_NAME>` содержит `run_config.json` и веса или адаптер.

Full FT загружается целиком; LoRA — как свежая исходная модель с сохранённым адаптером. Ревизия модели, исходные данные и постановка совпадают с обучением.

In [ ]:
RUN_NAME = 'base'
CHECKPOINT_KIND = 'base'
RUN_DIR = ROOT / 'runs' / RUN_NAME
assert not (RUN_DIR / 'evaluation.json').exists(), 'Оценка для этого опыта уже сохранена'

if CHECKPOINT_KIND == 'base':
    assert RUN_NAME == 'base'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    model = load_base_model(MODEL_NAME, revision=MODEL_REVISION, dtype=DTYPE, device=DEVICE)
else:
    trained_config = json.loads((RUN_DIR / 'run_config.json').read_text())
    assert trained_config['model_name'] == MODEL_NAME
    assert trained_config['model_revision'] == MODEL_REVISION
    tokenizer = AutoTokenizer.from_pretrained(RUN_DIR)
    if CHECKPOINT_KIND == 'full_ft':
        assert trained_config['lora'] is None
        model = AutoModelForCausalLM.from_pretrained(RUN_DIR, dtype=DTYPE).to(DEVICE)
    elif CHECKPOINT_KIND == 'lora':
        assert trained_config['lora'] is not None
        base_model = load_base_model(MODEL_NAME, revision=MODEL_REVISION, dtype=DTYPE, device=DEVICE)
        model = PeftModel.from_pretrained(base_model, RUN_DIR)
    else:
        raise ValueError('Ожидается base, full_ft или lora')
tokenizer.padding_side = 'left'

## Генерация и две метрики

`generate_answers` возвращает только продолжение после входного prompt. `rouge_l` оценивает ответы WikiWhy. `parse_choice` разбирает букву варианта, `history_accuracy` считает точность MMLU-Pro/history. Ошибка формата ответа остаётся ошибкой в общем знаменателе.

In [ ]:
wikiwhy_predictions = generate_answers(model, tokenizer, wikiwhy_prompts, GENERATION_CONFIG, batch_size=BATCH_SIZE)
history_predictions = generate_answers(model, tokenizer, history_prompts, GENERATION_CONFIG, batch_size=BATCH_SIZE)
assert len(wikiwhy_predictions) == len(wikiwhy_test)
assert len(history_predictions) == len(history_test)
choices = [parse_choice(prediction, row['options']) for prediction, row in zip(history_predictions, history_test)]
metrics = {
    'wikiwhy_rouge_l': rouge_l(wikiwhy_predictions, wikiwhy_references),
    'mmlu_history_accuracy': history_accuracy(choices, [row['answer'] for row in history_test]),
}
print(metrics)
print('Ошибок формата исторического ответа:', choices.count(None), '/', len(choices))

## Сохранение результатов

Ответы сохраняются с ID для сравнения одних и тех же вопросов всех моделей. Итоговая таблица читает сохранённые оценки. Разные протоколы используют отдельные имена опытов.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
prediction_record = {
    'wikiwhy': [{'id': row['id'], 'prediction': prediction} for row, prediction in zip(wikiwhy_test, wikiwhy_predictions)],
    'mmlu_history': [{'question_id': row['question_id'], 'prediction': prediction, 'choice': choice}
                     for row, prediction, choice in zip(history_test, history_predictions, choices)],
}
evaluation_record = {
    'metrics': metrics, 'generation': GENERATION_CONFIG,
    'wikiwhy_ids': [row['id'] for row in wikiwhy_test],
    'history_ids': [row['question_id'] for row in history_test],
    'history_parse_errors': choices.count(None),
}
with (RUN_DIR / 'predictions.json').open('x') as file:
    json.dump(prediction_record, file, ensure_ascii=False, indent=2)
with (RUN_DIR / 'evaluation.json').open('x') as file:
    json.dump(evaluation_record, file, ensure_ascii=False, indent=2)

## Итоговая таблица

Для base обучение отсутствует; пустые поля затрат не означают нулевое время инференса. Для Full FT и каждого LoRA-опыта таблица содержит число обучаемых параметров, время обучения и пик GPU-памяти. Конфигурации сохраняются рядом с результатами.

In [ ]:
rows = []
for path in sorted((ROOT / 'runs').glob('*/evaluation.json')):
    evaluation = json.loads(path.read_text())
    config_path, stats_path = path.parent / 'run_config.json', path.parent / 'train_stats.json'
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    stats = json.loads(stats_path.read_text()) if stats_path.exists() else {}
    rows.append({'run': path.parent.name, **evaluation['metrics'], **stats, **(config.get('lora') or {})})
results = pd.DataFrame(rows)
display(results)
# Итоговая таблица сохранённых оценок всех опытов.

## Ручная оценка 8 одинаковых исторических примеров

Восемь примеров выбираются с фиксированным seed из исторического среза WikiWhy test. Набор одинаков для base, Full FT и LoRA.

Критерии оценки: правильность причины, связность объясняющих переходов, хронология и неподтверждённые утверждения. Одна шкала применяется ко всем моделям. Разбор конкретных ошибок дополняет средние баллы.

In [ ]:
manual_rows = random.Random(42).sample([row for row in wikiwhy_test if row['topic'] == 'History'], 8)
manual = pd.DataFrame([{'id': row['id'], 'question': row['question'], 'reference': make_target(row)} for row in manual_rows])
for run_name in ('base', 'full_ft', 'lora_baseline'):
    path = ROOT / 'runs' / run_name / 'predictions.json'
    if path.exists():
        predictions = {row['id']: row['prediction'] for row in json.loads(path.read_text())['wikiwhy']}
        manual[run_name] = manual['id'].map(predictions)
display(manual)

## Анализ четырёх параметров LoRA

| Параметр | Значения | Изменение ROUGE-L | Изменение history accuracy | Изменения качества и затрат |
|---|---|---|---|---|
| r | | | | |
| lora_alpha | | | | |
| lora_dropout | | | | |
| target_modules | | | | |

Каждый вариант сравнивается с одним baseline при остальных фиксированных настройках. Анализ связывает конфигурацию с качеством, числом обучаемых параметров, памятью и временем. Небольшие различия сами по себе не устанавливают закономерность.

## Финальные выводы

Анализ результатов охватывает:

- изменения обеих метрик Full FT и LoRA относительно исходной модели;
- качество объяснений в ручной оценке и его связь с ROUGE-L;
- перенос дообучения на исторический экзамен;
- соотношение качества и вычислительных затрат;
- наиболее заметные эффекты четырёх параметров LoRA.

Выводы основываются на результатах выполненных опытов.